- [資料來源：高雄市歷年重大交通事故資料](https://data.gov.tw/dataset/47212)

假設你正在分析高雄歷年的重大交通事故，希望從資料中回答：
- 哪些事故原因最常出現？
- 哪種車種涉及重大事故最多？
- 哪個月份事故最多？
- 哪些事故造成的傷亡人數最高？

## 下載資料

In [5]:
import requests
import pandas as pd

### 設定 CSV 資料網址
# 高雄市歷年重大交通事故地點資料
url = "https://data.kcg.gov.tw/File/directDownload/996a1b2b-b5e1-4373-b368-626e6527d321"

headers = {
    "User-Agent": "Mozilla/5.0"
}

### 發送 HTTP 請求
response = requests.get(
    url,
    headers=headers,
    timeout=30
) # 向政府開放資料網站發送 GET 請求

response.raise_for_status() # 如果發生 4xx、5xx 錯誤，直接拋出例外

print("狀態碼：", response.status_code)


### 儲存 CSV
with open("../pandas_datasets/高雄市歷年重大交通事故地點資料.csv", "wb") as file:
    file.write(response.content) # 將伺服器回傳的 CSV 原始資料寫入檔案\

print("狀態碼：", response.status_code)
print("檔案大小：", len(response.content), "bytes")

狀態碼： 200
狀態碼： 200
檔案大小： 368 bytes


## 2. 讀取並查看資料

剛拿到陌生資料時，先看資料內容、大小、欄位名稱與資料型態。不要急著分析，因為欄位名稱和型態會直接影響後面的寫法。

In [6]:
import pandas as pd


### 讀取重大交通事故資料
df = pd.read_csv("../pandas_datasets/高雄市歷年重大交通事故地點資料.csv")


### 查看前 5 筆資料
df.head()

,Seq,ACCYMD,ACCTIME,PLACE,DEAD,HURT,CARTYPE,RSN
0,1,2010年1月3日,14:00,高雄縣內門鄉光興村台3線路段,7,2,廂型自小客貨車、營業貨運曳引車,車輛對撞
1,2,2011年6月26日,17:48,高雄市鳳山區國光路與五權南路口,1,18,自小客車,車輛撞行人
2,3,2013年2月15日,05:10,高雄市鳥松區美山路7號,4,1,自小客車,自撞


In [7]:
### 查看資料大小
print("資料列數：", df.shape[0])
print("資料欄數：", df.shape[1])


### 查看欄位名稱
print("欄位名稱：", df.columns.tolist())

資料列數： 3
資料欄數： 8
欄位名稱： ['Seq', 'ACCYMD', 'ACCTIME', 'PLACE', 'DEAD', 'HURT', 'CARTYPE', 'RSN']


In [8]:
### 查看資料摘要
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 8 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   Seq      3 non-null      int64
 1   ACCYMD   3 non-null      str  
 2   ACCTIME  3 non-null      str  
 3   PLACE    3 non-null      str  
 4   DEAD     3 non-null      int64
 5   HURT     3 non-null      int64
 6   CARTYPE  3 non-null      str  
 7   RSN      3 non-null      str  
dtypes: int64(3), str(5)
memory usage: 324.0 bytes


原始欄位使用英文縮寫，為了讓後面的分析更容易閱讀，先將欄位名稱改成中文。

| 原始欄位 | 中文欄位 | 說明 |
|---|---|---|
| `Seq` | 編號 | 資料序號 |
| `ACCYMD` | 事故日期 | 事故發生日期 |
| `ACCTIME` | 事故時間 | 事故發生時間 |
| `PLACE` | 事故地點 | 事故發生地點 |
| `DEAD` | 死亡人數 | 死亡人數 |
| `HURT` | 受傷人數 | 受傷人數 |
| `CARTYPE` | 車種 | 事故涉及的車種 |
| `RSN` | 事故原因 | 事故發生原因 |

In [9]:
### 修改欄位名稱
df = df.rename(columns={
    "Seq": "編號",
    "ACCYMD": "事故日期",
    "ACCTIME": "事故時間",
    "PLACE": "事故地點",
    "DEAD": "死亡人數",
    "HURT": "受傷人數",
    "CARTYPE": "車種",
    "RSN": "事故原因"
})

df.head()

,編號,事故日期,事故時間,事故地點,死亡人數,受傷人數,車種,事故原因
0,1,2010年1月3日,14:00,高雄縣內門鄉光興村台3線路段,7,2,廂型自小客貨車、營業貨運曳引車,車輛對撞
1,2,2011年6月26日,17:48,高雄市鳳山區國光路與五權南路口,1,18,自小客車,車輛撞行人
2,3,2013年2月15日,05:10,高雄市鳥松區美山路7號,4,1,自小客車,自撞


## 3. 檢查與整理資料

分析前要確認缺失值、重複值與欄位型態。日期欄位目前是文字，死亡與受傷人數也要保證是數值，否則後面無法正確計算月份和傷亡總數。

In [10]:
### 統計每個欄位的缺失值
missing_summary = df.isna().sum().to_frame(name="缺失值數量")
missing_summary

,缺失值數量
編號,0
事故日期,0
事故時間,0
事故地點,0
死亡人數,0
受傷人數,0
車種,0
事故原因,0


In [11]:
### 檢查重複資料
print("整列重複筆數：", df.duplicated().sum())
print("編號重複筆數：", df.duplicated(subset="編號").sum())

整列重複筆數： 0
編號重複筆數： 0


In [12]:
### 刪除缺少分析必要欄位的資料
required_columns = ["事故日期", "死亡人數", "受傷人數", "車種", "事故原因"]
df = df.dropna(subset=required_columns).copy()


### 轉換資料型態
date_text = df["事故日期"].str.replace("年", "-", regex=False)
date_text = date_text.str.replace("月", "-", regex=False)
date_text = date_text.str.replace("日", "", regex=False)

df["事故日期"] = pd.to_datetime(date_text, errors="coerce")
df["死亡人數"] = pd.to_numeric(df["死亡人數"], errors="coerce")
df["受傷人數"] = pd.to_numeric(df["受傷人數"], errors="coerce")


### 日期或人數轉換失敗時無法繼續分析，因此刪除
df = df.dropna(subset=["事故日期", "死亡人數", "受傷人數"]).copy()

df.dtypes

編號               int64
事故日期    datetime64[us]
事故時間               str
事故地點               str
死亡人數             int64
受傷人數             int64
車種                 str
事故原因               str
dtype: object

In [13]:
### 建立分析需要的新欄位
df["年份"] = df["事故日期"].dt.year
df["月份"] = df["事故日期"].dt.month
df["傷亡總人數"] = df["死亡人數"] + df["受傷人數"]


### 保留分析會使用的欄位
analysis_df = df[
    [
        "編號",
        "事故日期",
        "事故時間",
        "事故地點",
        "事故原因",
        "車種",
        "死亡人數",
        "受傷人數",
        "傷亡總人數",
        "年份",
        "月份"
    ]
].copy()

analysis_df

,編號,事故日期,事故時間,事故地點,事故原因,車種,死亡人數,受傷人數,傷亡總人數,年份,月份
0,1,2010-01-03,14:00,高雄縣內門鄉光興村台3線路段,車輛對撞,廂型自小客貨車、營業貨運曳引車,7,2,9,2010,1
1,2,2011-06-26,17:48,高雄市鳳山區國光路與五權南路口,車輛撞行人,自小客車,1,18,19,2011,6
2,3,2013-02-15,05:10,高雄市鳥松區美山路7號,自撞,自小客車,4,1,5,2013,2


## 4. 分析事故原因

`value_counts()` 可以統計每個事故原因出現幾次，再使用 `reset_index()` 將結果整理成 DataFrame。

In [14]:
### 統計各事故原因出現次數
reason_summary = analysis_df["事故原因"].value_counts().reset_index()
reason_summary.columns = ["事故原因", "事故次數"]

reason_summary

,事故原因,事故次數
0,車輛對撞,1
1,車輛撞行人,1
2,自撞,1


In [ ]:
### 找出最常出現的事故原因
max_reason_count = reason_summary["事故次數"].max()
top_reasons = reason_summary[reason_summary["事故次數"] == max_reason_count]

print("最常出現的事故原因：")
top_reasons
# 如果多個事故原因次數相同，直接取第一列會漏掉並列第一。因此先找最大次數，再用布林條件篩選所有並列結果。

最常出現的事故原因：


,事故原因,事故次數
0,車輛對撞,1
1,車輛撞行人,1
2,自撞,1




## 5. 分析事故車種

一場事故可能涉及多種車輛，例如「廂型自小客貨車、營業貨運曳引車」。如果直接統計原始文字，會把整串內容當成一種類別。這裡先用 `str.split("、")` 切開，再用 `explode()` 將每個車種拆成獨立一列。


In [17]:
### 將同一筆事故中的多個車種拆開
vehicle_data = analysis_df[["編號", "車種"]].copy()
vehicle_data["車種"] = vehicle_data["車種"].str.split("、")
vehicle_data = vehicle_data.explode("車種")
vehicle_data["車種"] = vehicle_data["車種"].str.strip()

vehicle_data


,編號,車種
0,1,廂型自小客貨車
0,1,營業貨運曳引車
1,2,自小客車
2,3,自小客車


In [18]:
### 統計各車種涉及重大事故的次數
vehicle_summary = vehicle_data["車種"].value_counts().reset_index()
vehicle_summary.columns = ["車種", "涉及事故次數"]

vehicle_summary


,車種,涉及事故次數
0,自小客車,2
1,廂型自小客貨車,1
2,營業貨運曳引車,1


In [20]:
### 找出涉及重大事故最多的車種
max_vehicle_count = vehicle_summary["涉及事故次數"].max()
top_vehicles = vehicle_summary[vehicle_summary["涉及事故次數"] == max_vehicle_count]

print("涉及重大事故最多的車種：")
top_vehicles


涉及重大事故最多的車種：


,車種,涉及事故次數
0,自小客車,2


## 6. 分析事故月份

前面已經從事故日期建立 `月份` 欄位，現在可以使用 `groupby()` 統計每個月份的事故筆數。沒有事故的月份不會出現在結果裡，使用 `reindex(range(1, 13), fill_value=0)` 可補齊 1 到 12 月。

In [22]:
### 統計各月份重大事故次數
month_summary = analysis_df.groupby("月份").size().reindex(range(1, 13), fill_value=0)
month_summary = month_summary.rename("事故次數").reset_index()

month_summary

,月份,事故次數
0,1,1
1,2,1
2,3,0
3,4,0
4,5,0
5,6,1
6,7,0
7,8,0
8,9,0
9,10,0


In [23]:
### 找出事故最多的月份
max_month_count = month_summary["事故次數"].max()
top_months = month_summary[month_summary["事故次數"] == max_month_count]

print("重大事故最多的月份：")
top_months

重大事故最多的月份：


,月份,事故次數
0,1,1
1,2,1
5,6,1


## 7. 分析傷亡人數

`傷亡總人數` 是死亡人數加上受傷人數。使用 `sort_values()` 由大到小排序，就能找出傷亡最嚴重的事故。


In [25]:
### 按照傷亡總人數由大到小排序
casualty_ranking = analysis_df[
    [
        "事故日期",
        "事故時間",
        "事故地點",
        "事故原因",
        "車種",
        "死亡人數",
        "受傷人數",
        "傷亡總人數"
    ]
].sort_values(
    ["傷亡總人數", "死亡人數"],
    ascending=[False, False]
)

casualty_ranking


,事故日期,事故時間,事故地點,事故原因,車種,死亡人數,受傷人數,傷亡總人數
1,2011-06-26,17:48,高雄市鳳山區國光路與五權南路口,車輛撞行人,自小客車,1,18,19
0,2010-01-03,14:00,高雄縣內門鄉光興村台3線路段,車輛對撞,廂型自小客貨車、營業貨運曳引車,7,2,9
2,2013-02-15,05:10,高雄市鳥松區美山路7號,自撞,自小客車,4,1,5


In [24]:
### 找出傷亡總人數最高的前 3 筆事故
top_3_casualties = analysis_df.nlargest(3, "傷亡總人數")
top_3_casualties = top_3_casualties[
    [
        "事故日期",
        "事故地點",
        "事故原因",
        "死亡人數",
        "受傷人數",
        "傷亡總人數"
    ]
]

top_3_casualties


,事故日期,事故地點,事故原因,死亡人數,受傷人數,傷亡總人數
1,2011-06-26,高雄市鳳山區國光路與五權南路口,車輛撞行人,1,18,19
0,2010-01-03,高雄縣內門鄉光興村台3線路段,車輛對撞,7,2,9
2,2013-02-15,高雄市鳥松區美山路7號,自撞,4,1,5
